In [1]:
import pandas as pd
import mwtab
import numpy as np
import seaborn as sns

# Crosswalks
* UNC RTI = __HHEAR Sample Inventory UNC-RTI.xlsx__
* mw_crosswalk = __ST002118_AN003466-SampleID Crosswalk.xlsx__
* hhear_crosswalk(meta_map) = __2254_META_MAP.csv__

In [2]:
unc_rti = r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Crosswalks\HHEAR Sample Inventory UNC-RTI.xlsx"
unc_rti = pd.read_excel(unc_rti)

mw_crosswalk = r'\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Crosswalks\ST002118_AN003466-SampleID Crosswalk.xlsx'
mw_crosswalk = pd.read_excel(mw_crosswalk)

META_MAP = pd.read_csv(r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Crosswalks\2254_META_MAP.csv")
META_MAP['timepoint_r'] = META_MAP['Sample_collection_timepoint'].map({"1st/2nd":"0","3rd":"1"})
META_MAP['CompositeKey'] = META_MAP['PID'].astype(str) + "_" + META_MAP['timepoint_r'].astype(str)

In [3]:
mw_crosswalk = mw_crosswalk[['Lab_ID','HHEAR_ID']]
meta_map = META_MAP[['PID','SID']].set_index("SID")
crosswalk = pd.merge(unc_rti,mw_crosswalk,left_on='CHEAR SSID',right_on='HHEAR_ID')
crosswalk['PID'] = crosswalk['CHEAR SSID'].map(meta_map['PID'])

In [4]:
crosswalk

,CHEAR PID,Original Subject ID,Sample ID,Sample Collection Date,Trimester,Working Box,R-C,Batch Box,CHEAR SSID,Lab_ID,HHEAR_ID,PID
0,6787451.0,N51127,N511272R2,2017-12-12,3rd,CHEAR SR 1,1-1,Batch 1,H-2GNY8-S-00,Lich_S_1,H-2GNY8-S-00,6787451
1,1586718.0,R73163,R731632R2,2017-12-05,3rd,CHEAR SR 1,1-2,Batch 1,H-2GP06-S-00,Lich_S_2,H-2GP06-S-00,1586718
2,6217673.0,R73244,R732442R2,2017-12-05,3rd,CHEAR SR 1,1-3,Batch 1,H-2GP14-S-00,Lich_S_3,H-2GP14-S-00,6217673
3,4044681.0,P40628,P406282R2,2017-10-25,3rd,CHEAR SR 1,1-4,Batch 1,H-2GP22-S-00,Lich_S_4,H-2GP22-S-00,4044681
4,3958659.0,P20272,P202722R2,2017-10-25,3rd,CHEAR SR 1,1-5,Batch 1,H-2GP30-S-00,Lich_S_5,H-2GP30-S-00,3958659
...,...,...,...,...,...,...,...,...,...,...,...,...
663,3166188.0,N51066,N510661R2,2017-05-22,1st/2nd,CHEAR SR 9,4-1,Batch 9,H-2H8W3-S-00,Lich_S_668,H-2H8W3-S-00,3166188
664,6664007.0,I62009,I620091R2,2017-05-29,1st/2nd,CHEAR SR 9,4-2,Batch 9,H-2H8U7-S-00,Lich_S_669,H-2H8U7-S-00,6664007
665,3958659.0,P20272,P202721R2,2017-05-10,1st/2nd,CHEAR SR 9,4-3,Batch 9,H-2H8J1-S-00,Lich_S_670,H-2H8J1-S-00,3958659
666,7285182.0,R73062,R730621R2,2017-05-30,1st/2nd,CHEAR SR 9,4-4,Batch 9,H-2H8H6-S-00,Lich_S_671,H-2H8H6-S-00,7285182


## Untargeted Data From Metabolomics Workbench (ST002118)

In [5]:
untargeted_data = pd.read_table(r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Untargeted\ST002118_AN003466_Results.txt",sep='\t',header=0)
untargeted_data = untargeted_data.rename(columns={"RT_mass":'SID_MW'})
untargeted_data.set_index("SID_MW",inplace=True)
untargeted_data = untargeted_data.T.reset_index()
untargeted_data.rename(columns={"index":"SID_MW"},inplace=True)
untargeted_data.rename_axis("",axis=1,inplace=True)
untargeted_data.set_index("SID_MW",inplace=True)
trimester_id = crosswalk[["Lab_ID","Trimester",'PID']].set_index("Lab_ID")
untargeted_data = pd.merge(untargeted_data,trimester_id,left_index=True,right_index=True)

In [6]:
cols = untargeted_data.select_dtypes(include="float").filter(regex=r"(m/z|n)$").columns
cols_to_drop = cols[untargeted_data[cols].sum() == 0]
untargeted_data = untargeted_data.drop(columns=cols_to_drop)
untargeted_data

,0.03_183.9122m/z,0.03_98.9842m/z,0.11_132.9982m/z,0.11_279.1589m/z,0.13_113.0233m/z,0.41_113.0209m/z,0.48_102.0117m/z,0.48_103.9960n,0.48_104.9835m/z,0.48_104.9992n,...,9.96_289.0956m/z,9.96_301.1493m/z,9.96_314.1262n,9.96_321.1200m/z,9.96_323.0967m/z,9.96_332.0908n,9.96_337.0498m/z,9.98_2355.2210n,Trimester,PID
Lich_S_1,1635.909235,2355.780161,1319.285570,16089.652920,3106.032673,1596.678462,359.331388,285231.5814,407.154777,4814.714080,...,3402.748962,19085.148980,14781.997790,5518.091497,15221.574340,183731.85940,9013.530274,1171.794555,3rd,6787451
Lich_S_2,1561.119350,2404.144409,1284.219196,4335.110276,2905.345460,1478.895423,399.763628,342173.5893,433.477860,5934.687589,...,2591.741892,14328.121930,11970.298430,4624.573472,11527.488230,160424.49910,8029.943005,2243.664265,3rd,1586718
Lich_S_3,2324.998517,3291.565605,1792.701122,5163.360856,4351.107792,2020.605138,704.096546,615687.6507,817.726335,10230.542300,...,1985.743563,15611.511150,12846.190500,3605.101386,9528.032313,126779.96740,7464.842752,95856.974980,3rd,6217673
Lich_S_4,1779.616397,2585.441632,1417.371073,2792.335932,3570.893062,1683.844514,335.728084,461586.8015,710.203282,8170.253220,...,2666.719412,16556.791810,12872.709110,5690.526836,13185.125230,156547.20950,7451.371760,9726.200966,3rd,4044681
Lich_S_5,1190.537830,1697.403779,972.953378,1598.097538,2488.120860,1085.926642,153.921614,165028.3337,251.210193,2667.483654,...,2167.651160,14007.758410,12038.417360,5604.767976,12378.694500,179007.99390,8171.752064,5403.001413,3rd,3958659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Lich_S_668,2091.776006,2295.282200,1493.544702,1724.085946,2496.697921,1780.238613,238.939717,195807.7494,300.514404,3005.883865,...,2689.134614,11986.471520,15619.066910,3519.699074,7197.635440,130292.43350,6842.076107,2012.408701,1st/2nd,3166188
Lich_S_669,2105.218078,2331.863922,1567.297993,1309.636011,2090.988137,1833.653672,457.167059,393017.2979,453.040463,6266.091118,...,2685.930682,9732.256540,5557.740578,2363.389000,4808.621532,91883.32741,6088.963357,9076.852598,1st/2nd,6664007
Lich_S_670,1555.798241,1749.363529,1176.852203,1292.012700,1674.875305,1640.520555,232.774629,204561.1121,270.351494,3364.992579,...,3253.599483,13660.437620,14411.365420,3614.754364,7787.409255,139273.84010,7454.384849,4252.256342,1st/2nd,3958659
Lich_S_671,1801.371333,2359.088506,1398.308769,2546.001781,2059.329332,1615.934476,277.126931,217295.0191,238.109426,3285.131226,...,3706.687454,9640.792742,6496.271729,2899.578320,5268.164965,110343.90180,6515.883944,2202.903279,1st/2nd,7285182


In [ ]:
#untargeted_data.to_csv(r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Untargeted\ST002118_Untargeted_Reference_Copy.csv")

## Targeted Data From HHEAR Data Center (ST002118) (Converted to Wide Format)

In [220]:
file = r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Targeted\2254_TARGETED_DATA (2).xlsx"
targeted_data = pd.read_excel(file)
targeted_data['timepoint_r'] = targeted_data['sample_collection_timepoint'].map({"1st/2nd":0,"3rd":1})
targeted_data['CompositeKey'] = targeted_data['PID'].astype(str) + "_" + targeted_data['timepoint_r'].astype(str)


# Select all Valid Measurements and replace Below LOD with LOD / sqrt(2)
targeted_data = targeted_data[~targeted_data['Comment_code'].isin([18,19,129])]
blod_mask = targeted_data['Comment_code'].isin([37, 201])

targeted_data.loc[blod_mask, 'Concentration'] = (
    targeted_data.loc[blod_mask, 'LOD'] / np.sqrt(2)
)

In [221]:
targeted_wide = targeted_data.pivot_table(
    columns='Analyte_Code',
    values='Concentration',
    index='CompositeKey',
    aggfunc='first'
)

In [222]:
sample_ids = META_MAP[["SID","SID_MW"]].set_index("SID_MW")

In [225]:
meta_map = {}
for index,row in META_MAP.iterrows():
    meta_map[str(row['CompositeKey'])] = str(row['SID_MW']).strip()
targeted_wide['SID_MW'] = targeted_wide.index.map(meta_map)
targeted_wide = targeted_wide.dropna(subset=['SID_MW'])
targeted_wide['Trimester_r'] = targeted_wide.index.str.split("_").str[1]
targeted_wide['Trimester_r']= targeted_wide['Trimester_r'].map({"0":"1st/2nd","1":"3rd"})
targeted_wide = targeted_wide.rename(columns={"Cd":"Cd (ng/mL)",
                              "Mn":"Mn (ng/mL)",
                              "Se":"Se (ng/mL)",
                              "Pb":"Pb (ug/dL)",
                              "Hg":"Hg (ng/mL)",
                              "Sn":"Sn (ng/mL)"})
targeted_wide['PID'] = targeted_wide.index.str.split("_").str[0]
targeted_wide['HHEAR'] = targeted_wide["SID_MW"].map(sample_ids['SID'])
targeted_wide = targeted_wide.reset_index(drop=True).set_index("SID_MW")

In [ ]:
#targeted_wide.to_parquet(r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Targeted\ST002118_TARGETED_REFERENCE_WIDE_071426.parquet")

In [240]:
targeted_wide

Analyte_Code,Cd (ng/mL),Hg (ng/mL),Mn (ng/mL),Pb (ug/dL),Se (ng/mL),Sn (ng/mL),Trimester_r,PID,HHEAR
SID_MW,,,,,,,,,
Lich_S_69,0.12,5.76,6.8,0.70,138.0,0.424264,1st/2nd,1029768,H-2GR12-S-00
Lich_S_354,0.12,5.91,9.4,0.76,111.0,0.424264,3rd,1029768,H-2GZR2-S-00
Lich_S_449,0.25,4.42,11.3,5.40,85.7,1.080000,1st/2nd,1102367,H-2H1R1-S-00
Lich_S_484,0.28,3.24,18.2,3.30,70.0,0.970000,3rd,1102367,H-2H5J4-S-00
Lich_S_627,0.11,1.05,6.6,1.02,115.0,0.424264,1st/2nd,1111293,H-2H824-S-00
...,...,...,...,...,...,...,...,...,...
Lich_S_383,0.17,4.85,17.6,8.61,126.0,0.424264,1st/2nd,9818485,H-2H162-S-00
Lich_S_127,0.18,4.24,24.2,3.40,117.0,0.424264,3rd,9818485,H-2GSR9-S-00
Lich_S_614,0.13,2.23,12.8,1.29,109.0,0.424264,1st/2nd,9868688,H-2H6X2-S-00


# EPI Data From HHEAR Data Center

In [270]:
epi_data = pd.read_csv(r'\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\EPI_DATA\2254_EPI_DATA.csv')
epi_data = epi_data.set_index("PID")

In [273]:
epi_data.loc[untargeted_data['PID'].drop_duplicates()][['v107', 'v110', 'v111a', 'v111b', 'v111c', 'v111d',
                                                        'v112a', 'v113c', 'v115a', 'v118a', 'v118b', 'v140n1', 'v140n3',
                                                        'v150n1', 'v150n2', 'v207', 'v240n1', 'v240n3', 'v250n1', 'v250n2',
                                                        'b031', 'b031a', 'b034b', 'b034c', 'b035b', 'b035d', 'b041']]

,v107,v110,v111a,v111b,v111c,v111d,v112a,v113c,v115a,v118a,...,v240n3,v250n1,v250n2,b031,b031a,b034b,b034c,b035b,b035d,b041
PID,,,,,,,,,,,,,,,,,,,,,
6787451,23.014374,2,169.0,65.0,22.758307,2.0,3,6,5,0,...,1.0,3.0,2.0,3170.0,2.0,38.0,2.0,10.0,2.0,23.266256
1586718,28.383299,2,154.0,82.0,34.575814,4.0,3,1,1,5,...,2.0,14.0,1.0,3010.0,2.0,38.4,2.0,9.0,2.0,19.627652
6217673,32.128679,1,152.0,55.0,23.805402,2.0,1,5,3,2,...,1.0,10.0,2.0,3020.0,2.0,38.4,2.0,10.0,2.0,32.383299
4044681,34.551677,1,164.0,82.0,30.487805,4.0,7,5,6,0,...,2.0,9.0,2.0,3320.0,2.0,40.0,2.0,10.0,2.0,35.140315
3958659,27.600274,1,165.0,55.0,20.202020,2.0,6,2,4,2,...,1.0,4.0,2.0,3640.0,2.0,41.2,2.0,10.0,2.0,28.208077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7574932,24.884326,1,150.9,80.4,35.308362,4.0,6,2,4,0,...,2.0,12.0,1.0,3105.0,2.0,38.4,2.0,10.0,2.0,24.887064
3509379,31.865845,2,165.0,97.0,35.629017,4.0,4,1,6,0,...,2.0,9.0,2.0,3855.0,2.0,39.3,2.0,10.0,2.0,32.383299
6497705,30.839151,1,166.0,75.0,27.217303,3.0,6,2,3,3,...,1.0,4.0,2.0,3165.0,2.0,39.0,2.0,9.0,2.0,31.175907


In [ ]:
#wick = pd.read_parquet(r'\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026\Data\Untargeted\WicklifeDataset\reference_copy_categorical_data.parquet')